# ViFeedback — Kaggle training (robust version)

This notebook is a thin Kaggle runner for the `vifeedback` repository. It supports both common
Kaggle input layouts:

- a dataset containing a repository **ZIP**; or
- a Kaggle dataset where the repository files are already **extracted** under `/kaggle/input/...`.

Results still use the repository's own `results/registry.csv` schema.

## Before you run anything

| Step | Where |
|---|---|
| 1. Accelerator → **GPU** | *Settings* panel, right side |
| 2. Internet → **On** | Required for Hugging Face/data downloads unless everything is cached |
| 3. Attach the repo dataset | *Add Input* → your `vifeedback-repo` dataset |
| 4. *(optional)* `HF_TOKEN` secret | *Add-ons → Secrets* |

> The notebook no longer assumes the attached dataset must contain a `.zip`; it auto-detects the
> repository by locating `pyproject.toml`.

## What belongs here

Use Kaggle for models that do not fit comfortably on the reference laptop GPU. Do **not** use
Kaggle latency numbers as laptop/reference-machine latency numbers because the hardware differs.


---

## 1. Verify the environment


In [1]:
import shutil
import socket
import subprocess
import torch

# GPU diagnostics: do not fail just because nvidia-smi formatting changes.
if shutil.which('nvidia-smi'):
    subprocess.run([
        'nvidia-smi',
        '--query-gpu=name,memory.total,driver_version',
        '--format=csv,noheader',
    ], check=False)

assert torch.cuda.is_available(), (
    'No CUDA GPU is available. In Kaggle: Settings -> Accelerator -> select a GPU, '
    'then restart/re-run the notebook.'
)

p = torch.cuda.get_device_properties(0)
vram_gb = p.total_memory / (1024**3)
print(f'{p.name}  {vram_gb:.2f} GiB  capability {p.major}.{p.minor}  torch {torch.__version__}')

# The planned models need substantially more than a 4 GB laptop GPU. Keep this as a warning
# instead of a hard assertion so the notebook can still be used for smaller/debug runs.
if vram_gb < 10:
    print(f'WARNING: only {vram_gb:.1f} GiB VRAM; large-model cells may OOM. '
          'Reduce batch size / use grad accumulation or choose a larger Kaggle GPU.')

try:
    socket.create_connection(('huggingface.co', 443), timeout=5).close()
    print('internet: OK')
except OSError:
    print('WARNING: huggingface.co is not reachable. Hub/data download steps may fail. '
          'Enable Internet in Kaggle Settings unless all required assets are already cached.')


Tesla T4, 15360 MiB, 580.159.04
Tesla T4, 15360 MiB, 580.159.04
Tesla T4  14.56 GiB  capability 7.5  torch 2.10.0+cu128
internet: OK


### Optional: Hugging Face token

Unauthenticated Hub downloads are rate-limited. If you added an `HF_TOKEN` secret
(*Add-ons → Secrets*), this picks it up; if not, it carries on unauthenticated.


In [2]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    print(f'No HF_TOKEN ({type(e).__name__}) — continuing unauthenticated, which is fine')

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['PYTHONUNBUFFERED'] = '1'


No HF_TOKEN (BackendError) — continuing unauthenticated, which is fine


---

## 2. Prepare the repository

The cell below supports both repository formats used by Kaggle:

1. **Extracted dataset**: `/kaggle/input/<dataset>/pyproject.toml`, `src/`, `tests/`, ...
2. **ZIP dataset**: a `.zip` somewhere under `/kaggle/input`.

If `REPO_URL` is set, cloning takes precedence. Otherwise the notebook searches for an extracted
repository first, then falls back to a ZIP.


In [3]:
import glob
import os
import pathlib
import shutil

REPO_URL = ''  # Route A, e.g. 'https://github.com/<user>/ViFeedback-NLP-Service.git'

# NOT named 'vifeedback'. A directory of that name on sys.path becomes a PEP 420
# namespace package and shadows the installed library: `vifeedback.data` then resolves
# to the dataset FOLDER and `vifeedback.evaluation` raises ModuleNotFoundError - which
# is exactly what killed a 40-minute run on this notebook's previous version.
WORK = pathlib.Path('/kaggle/working/repo')
assert WORK.name != 'vifeedback', 'this directory name shadows the package'

if WORK.exists():
    shutil.rmtree(WORK)

if REPO_URL:
    !git clone -q {REPO_URL} {WORK}
else:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        print('using zip:', zips[0])
        WORK.mkdir(parents=True)
        !unzip -qo {zips[0]} -d {WORK}
    else:
        # Kaggle auto-extracts some dataset uploads; fall back to copying the tree.
        cands = [pathlib.Path(d).parent for d in
                 glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)]
        assert cands, ('No repo found under /kaggle/input. Add Data -> attach your '
                       'vifeedback-repo dataset. See docs/KAGGLE_GUIDE.md step 1.')
        print('using extracted repo:', cands[0])
        shutil.copytree(cands[0], WORK)

os.chdir(WORK)
assert (WORK / 'pyproject.toml').exists(), f'pyproject.toml missing in {WORK}'
assert (WORK / 'src' / 'vifeedback' / 'evaluation').is_dir(), (
    'src/vifeedback/evaluation is missing - the uploaded dataset is a STALE build. '
    'Rebuild it with `git archive --format=zip -o vifeedback.zip HEAD` and upload a New '
    'Version of the dataset.'
)
print('cwd:', pathlib.Path.cwd())
print('top-level:', sorted(x.name for x in WORK.iterdir()))


Using extracted repository: /kaggle/input/datasets/datthnh/vifeedback-repo
cwd: /kaggle/working/vifeedback
top-level: ['.gitignore', 'README.md', 'data', 'docs', 'notebooks', 'pyproject.toml', 'results', 'src', 'tests']


### Install

`--no-deps` on the project itself: Kaggle already ships torch, numpy, pandas and sklearn, and
letting pip resolve `pyproject.toml` in full can pull a different torch and break CUDA.
The handful of packages Kaggle lacks are installed explicitly.


In [4]:
import subprocess
import sys

# --no-deps first so pip cannot replace Kaggle's CUDA torch.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'typer', 'pyyaml', 'py-cpuinfo', 'pyvi', 'underthesea', 'pytest',
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers>=4.44', 'datasets>=3.0', 'huggingface-hub',
], check=True)

import vifeedback

# check_import() RAISES if `import vifeedback` bound to a PEP 420 namespace package instead
# of the library. The previous version of this cell printed '(no __version__)' and carried on,
# so the run proceeded for 40 minutes and then died on `vifeedback.evaluation` with a
# traceback that named neither the cause nor the fix. Failing here costs five seconds.
print('vifeedback', vifeedback.check_import())
print('resolved to:', vifeedback.__file__)

# Import the modules the training cells need, NOW, while a failure is cheap.
from vifeedback.evaluation.report import load_registry  # noqa: F401
from vifeedback.training import TrainConfig  # noqa: F401

subprocess.run([sys.executable, '-m', 'vifeedback.cli', '--help'], check=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 107.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


vifeedback (no __version__)
                                                                                
 Usage: python -m vifeedback.cli [OPTIONS] COMMAND [ARGS]...                    
                                                                                
 ViFeedback — Vietnamese feedback classification                                
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help          Show this message and exit.                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ data       Phase 0: acquisition, integrity, profiling                        │
│ baseline   Phase 1: classical baselines                                      │
│ train      Phase 2+: transformer fine-tuning                                 │


CompletedProcess(args=['/usr/bin/python3', '-m', 'vifeedback.cli', '--help'], returncode=0)

---

## 3. Data

Fetches UIT-VSFC and asserts the official split sizes (11,426 / 1,583 / 3,166) plus the leakage
figures. If the upstream ever shifts, this fails here rather than producing numbers against
different data.


In [5]:
import subprocess
import sys

# Run through subprocess so failures stop the cell immediately (plain !commands can make it easy
# to miss an earlier non-zero exit code in a long output stream).
subprocess.run([sys.executable, '-m', 'vifeedback.cli', 'data', 'fetch'], check=True)

TESTS_DATA = WORK / 'tests' / 'data'
if TESTS_DATA.exists():
    subprocess.run([sys.executable, '-m', 'pytest', str(TESTS_DATA), '-q'], check=True)
else:
    print(f'WARNING: {TESTS_DATA} does not exist; skipping repository data tests.')


  train       11,426 rows  sha256=c0fdba92766e2cb3...
  validation   1,583 rows  sha256=2c85203815cb2d5c...
  test         3,166 rows  sha256=a2b7beaa5a6ff745...
  source: uitnlp/vietnamese_students_feedback@refs/convert/parquet
..............                                                           [100%]


Build the segmentation variant. Segmentation is worth **+0.023 macro-F1** (ADR-012), so training
on raw text here would not be comparable to the laptop results. `pyvi` is the serving choice and
is pure Python, so no JVM is needed.


In [6]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'vifeedback.cli', 'data', 'segmenters'], check=True)
subprocess.run([
    sys.executable, '-m', 'vifeedback.cli', 'data', 'variants', '--name', 'seg_pyvi'
], check=True)


  OK  none           identity
  --  underthesea    underthesea not installed
  OK  pyvi           importable
  --  vncorenlp      py_vncorenlp not installed
variant                 cond  changed%  underscores  tok.reduction  ms/sentence
-------------------------------------------------------------------------------
seg_pyvi                 P2b    92.0%       35,178         21.5%        0.178


CompletedProcess(args=['/usr/bin/python3', '-m', 'vifeedback.cli', 'data', 'variants', '--name', 'seg_pyvi'], returncode=0)

---

## 4. Train

**Runtime estimates** (P100, 5 seeds, 4 epochs). Kaggle sessions run up to 9 h and the free quota
is 30 GPU-hours/week, so run the cells you need rather than all of them.

| Cell | Model | Est. total |
|---|---|---|
| 4a | `phobert-large` | **~75 min** |
| 4b | `xlm-roberta-base` full | ~40 min |
| 4c | `CafeBERT` | ~90 min |

### On `--grad-accum`

`phobert-large` needs batch 16 to fit comfortably, but `phobert-base` was trained at 32. Comparing
them at different effective batch sizes would not be a controlled comparison, so `--grad-accum 2`
restores the effective batch to 32. The CLI prints the effective batch it is using — check it.


In [7]:
# 4a — PhoBERT-large. The main reason this notebook exists (~7.9 GB).
!python -m vifeedback.cli train run \
    --task sentiment \
    --model phobert-large \
    --recipe base \
    --preprocessing seg_pyvi \
    --seeds all \
    --epochs 4 \
    --lr 1e-5 \
    --batch-size 16 \
    --grad-accum 2 \
    --max-length 96 \
    --phase 4


  effective batch = 16 x 2 = 32

[p4-sent-phobert-large-seg_pyvi-base-s42-val]  phobert-large / sentiment / base / seed 42
config.json: 100%|█████████████████████████████| 558/558 [00:00<00:00, 2.90MB/s]
vocab.txt: 100%|█████████████████████████████| 895k/895k [00:00<00:00, 23.3MB/s]
bpe.codes: 100%|████████████████████████████| 1.14M/1.14M [00:00<00:00, 110MB/s]
tokenizer.json: 100%|███████████████████████| 3.13M/3.13M [00:00<00:00, 169MB/s]

pytorch_model.bin: downloading bytes: ██                    |  143MB, 10.6MB/s  
pytorch_model.bin: downloading bytes: ████                  |  277MB, 22.1MB/s  
pytorch_model.bin: downloading bytes: ████▊                 |  323MB, 25.9MB/s  
pytorch_model.bin: downloading bytes: ██████▉               |  466MB, 37.7MB/s  
pytorch_model.bin: downloading bytes: ██████████████▎       |  967MB, 80.7MB/s  
pytorch_model.bin: reconstructing file:  69%|████▏ | 1.02GB / 1.48GB, 49.1MB/s  
pytorch_model.bin: downloading bytes: ██████████████████████|  995

In [8]:
# 4b — XLM-R base with UNFROZEN embeddings (~5.9 GB).
# The frozen-embedding version fits the laptop; run that one there, not here.
!python -m vifeedback.cli train run \
    --task sentiment \
    --model xlmr-base \
    --recipe base \
    --preprocessing seg_pyvi \
    --seeds all \
    --epochs 4 \
    --batch-size 32 \
    --max-length 96 \
    --phase 4


  effective batch = 32 x 1 = 32

[p4-sent-xlmr-base-seg_pyvi-base-s42-val]  xlmr-base / sentiment / base / seed 42
config.json: 100%|█████████████████████████████| 615/615 [00:00<00:00, 3.93MB/s]
tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 169kB/s]
sentencepiece.bpe.model: 100%|█████████████| 5.07M/5.07M [00:00<00:00, 59.2MB/s]
tokenizer.json: 100%|███████████████████████| 9.10M/9.10M [00:00<00:00, 116MB/s]

model.safetensors: downloading bytes: ████▎                 |  219MB, 17.1MB/s  
model.safetensors: downloading bytes: ████████▊             |  446MB, 36.5MB/s  
model.safetensors: downloading bytes: █████████████▉        |  705MB, 59.2MB/s  
model.safetensors: reconstructing file:  92%|█████▌| 1.02GB / 1.12GB, 24.9MB/s  
model.safetensors: downloading bytes: ██████████████████████|  745MB, 64.8MB/s  
model.safetensors: reconstructing file: 100%|██████| 1.12GB / 1.12GB,  101MB/s  
Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 2739.9

In [9]:
# 4c — CafeBERT (XLM-R-large continued on 18GB Vietnamese, 560M params, ~11 GB).
# Needs registering in constants.MODEL_IDS first; skip unless you have added it.
# !python -m vifeedback.cli train run --task sentiment --model cafebert \
#     --preprocessing seg_pyvi --seeds all --batch-size 8 --grad-accum 4 --lr 8e-6 --phase 4


---

## 5. Verify before you leave

Confirms runs actually landed in the registry. A session that finished without writing rows has
produced nothing, and it is much cheaper to notice that here than after the session expires.


In [10]:
import pathlib
import pandas as pd

from vifeedback.evaluation.report import load_registry

reg = load_registry()
print(f'total registry rows: {len(reg)}')

# Be robust to differing registry schemas: prefer the explicit model column, and fall back to run_id.
model_pattern = r'phobert-large|xlmr-base|xlmr|cafebert'
mask = pd.Series(False, index=reg.index)
if 'model' in reg.columns:
    mask |= reg['model'].astype(str).str.contains(model_pattern, case=False, regex=True, na=False)
if 'run_id' in reg.columns:
    mask |= reg['run_id'].astype(str).str.contains(model_pattern, case=False, regex=True, na=False)

new = reg[mask].copy()
print(f'rows matching Kaggle-target models: {len(new)}')

if len(new) == 0:
    print('WARNING: no registry rows matched the expected model names. '
          'If training completed, inspect the registry columns/values below before assuming data was lost.')
    display(reg.tail(min(20, len(reg))))
else:
    preferred = ['run_id', 'task', 'model', 'split', 'macro_f1', 'weighted_f1', 'accuracy']
    display(new[[c for c in preferred if c in new.columns]].tail(50))


ModuleNotFoundError: No module named 'vifeedback.evaluation'

In [ ]:
# Mean +/- std per configuration when the expected columns exist.
required = {'task', 'model', 'split', 'macro_f1'}
if len(new) and required.issubset(new.columns):
    val = new[new['split'].astype(str).str.lower().eq('validation')]
    if len(val):
        display(
            val.groupby(['task', 'model'])['macro_f1']
            .agg(['count', 'mean', 'std', 'min', 'max'])
            .round(4)
        )
    else:
        print('No validation rows found among the matched runs.')
else:
    print('Summary skipped because the registry does not contain all required columns:', sorted(required))


---

## 6. Bring the results home

`/kaggle/working` is saved with the notebook version, so **Save Version → Output** already
persists everything. The zip below is for downloading it as one file.


In [ ]:
import pathlib
import shutil

results_dir = WORK / 'results'
assert results_dir.exists(), (
    f'{results_dir} does not exist. Run the training/evaluation cells first, or check where '
    'the repository writes its results.'
)

archive_base = pathlib.Path('/kaggle/working/kaggle_results')
archive_path = pathlib.Path(
    shutil.make_archive(str(archive_base), 'zip', root_dir=str(results_dir))
)
size_mb = archive_path.stat().st_size / 1e6
print(f'{archive_path}  ({size_mb:.1f} MB)')
print('Download it from the Output panel on the right after the cell finishes.')


### Merging on the laptop

Merge **by `run_id`**, never by blind append — the archive's `registry.csv` also contains rows
that were already committed before you built the dataset.

```bash
unzip -o kaggle_results.zip -d /tmp/kag
cp -rn /tmp/kag/runs/* results/runs/
python - <<'EOF'
import pandas as pd
a = pd.read_csv('results/registry.csv')
b = pd.read_csv('/tmp/kag/registry.csv')
out = pd.concat([a, b]).drop_duplicates(subset='run_id', keep='first')
out.to_csv('results/registry.csv', index=False)
print(f'{len(out) - len(a)} new rows merged')
EOF
```

### Reporting the split honestly

Rows produced here carry a different `env.json` — P100, not RTX 3050. That is irrelevant for
**accuracy**, which is hardware-independent, and disqualifying for **latency**, which is not.
Mark Kaggle-trained rows in the final results table and name the GPU.
